# 🎮 Face Recognition Game: Colab 2D Edition
**พิเศษสุดๆ! เวอร์ชันรันบน Google Colab ได้ทันที พร้อมกราฟิกสไตล์เกม 2D**
- แสดงผลในหน้าเว็บ Colab เลย!
- เล่นผ่านกล้องเว็บแคมได้ลื่นๆ
- มีเพลงประกอบ (อัปโหลดไฟล์ g_game.mp3 ขึ้น Colab)
- เมนูตั้งค่าสุดอลังการแบบ 2D UI


In [ ]:
# 1. ติดตั้งไลบรารีที่จำเป็น
# !pip install facenet-pytorch scikit-learn joblib Pillow numpy


In [ ]:
# 2. นำเข้าไลบรารีและโหลดโมเดล
import base64
import json
import time
import cv2
import numpy as np
import random
import os
import torch
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
import joblib
from google.colab import output
from IPython.display import display, HTML

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'💻 ใช้หน่วยประมวลผล: {device}')

print('กำลังโหลดโมเดล FaceNet...')
mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, select_largest=False, post_process=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

clf_path = 'facenet_svm_model.pkl'
le_path = 'label_encoder.pkl'

if os.path.exists(clf_path) and os.path.exists(le_path):
    clf = joblib.load(clf_path)
    le = joblib.load(le_path)
    all_names = list(le.classes_)
    print(f'✅ โหลดโมเดลสำเร็จ! รายชื่อในระบบ: {all_names}')
else:
    raise FileNotFoundError('❌ ไม่พบไฟล์โมเดล! กรุณาอัปโหลด facenet_svm_model.pkl และ label_encoder.pkl ขึ้น Colab ก่อน')


In [ ]:
# 3. รันเพื่อเปิดหน้าจอเกม 2D บนเว็บ Colab!
def start_colab_game(all_names, mtcnn, resnet, clf, le, device):
    game_state = {
        'score': 0, 'time_limit': 60, 'start_time': 0, 'target_name': '',
        'active_names': all_names, 'confidence_needed': 0.90, 'game_over': False, 'playing': False
    }

    def start_game_py(total_time, diff, names_str):
        game_state['score'] = 0
        game_state['time_limit'] = float(total_time)
        game_state['start_time'] = time.time()
        game_state['game_over'] = False
        game_state['playing'] = True
        
        if diff == '1': game_state['confidence_needed'] = 0.80
        elif diff == '3': game_state['confidence_needed'] = 0.95
        else: game_state['confidence_needed'] = 0.90
        
        if names_str.strip() == "": game_state['active_names'] = all_names
        else:
            names = [n.strip() for n in names_str.split(',')]
            game_state['active_names'] = [n for n in names if n in all_names] or all_names
            
        game_state['target_name'] = random.choice(game_state['active_names'])
        return "OK"

    output.register_callback('start_game_py', start_game_py)

    def process_frame(b64_str):
        if not game_state['playing'] or game_state['game_over']: return json.dumps({'game_over': True, 'score': game_state['score']})
            
        img_data = base64.b64decode(b64_str.split(',')[1])
        nparr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img = cv2.flip(img, 1)
        
        elapsed = time.time() - game_state['start_time']
        time_left = max(0, game_state['time_limit'] - elapsed)
        if time_left <= 0:
            game_state['game_over'] = True
            return json.dumps({'game_over': True, 'score': game_state['score']})
            
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        boxes, probs = mtcnn.detect(pil_img)
        
        faces_data = []
        highest_conf = 0.0
        passed = False
        
        if boxes is not None:
            faces = mtcnn(pil_img)
            if faces is not None:
                faces_tensor = faces.to(device)
                if faces_tensor.dim() == 3: faces_tensor = faces_tensor.unsqueeze(0)
                
                with torch.no_grad(): embeddings = resnet(faces_tensor).cpu().numpy()
                preds = clf.predict(embeddings)
                pred_probs = clf.predict_proba(embeddings)
                
                for i, box in enumerate(boxes):
                    if i >= len(preds) or probs[i] < 0.90: continue
                    w, h = box[2] - box[0], box[3] - box[1]
                    if w < 40 or h < 40: continue
                    
                    prob_max = float(np.max(pred_probs[i]))
                    name = str(le.inverse_transform([preds[i]])[0])
                    if name == game_state['target_name']:
                        if prob_max > highest_conf: highest_conf = prob_max
                    
                    faces_data.append({'box': [float(x) for x in box], 'name': name, 'prob': prob_max})
                    
        if highest_conf >= game_state['confidence_needed']:
            game_state['score'] += 1
            game_state['target_name'] = random.choice(game_state['active_names'])
            passed = True
            
        return json.dumps({
            'game_over': False, 'time_left': time_left, 'score': game_state['score'],
            'target_name': game_state['target_name'], 'confidence_needed': game_state['confidence_needed'],
            'highest_conf': highest_conf, 'passed': passed, 'faces': faces_data
        })

    output.register_callback('process_frame_game', process_frame)

    audio_data = ""
    if os.path.exists('bg_game.mp3'):
        with open('bg_game.mp3', 'rb') as f: audio_data = f"data:audio/mp3;base64,{base64.b64encode(f.read()).decode()}"

    html_code = f'''
    <div id="game-container" style="background: linear-gradient(135deg, #111, #2a0845); padding:20px; text-align:center; border-radius:15px; font-family: 'Segoe UI', Tahoma, sans-serif; color:white; width:680px; margin:0 auto; box-shadow: 0 0 30px rgba(255,0,255,0.4); border: 2px solid #ff00ff;">
        <h1 style="color:#0ff; text-shadow: 0 0 10px #0ff, 2px 2px 0px #000; margin-bottom: 20px; font-size: 36px;">🤖 FACE HERO 🤖</h1>
        <audio id="bgm" src="{audio_data}" loop></audio>
        <div id="setup-menu" style="background: rgba(0,0,0,0.6); padding: 20px; border-radius: 10px; border: 1px solid #0ff; text-align: left; width: 80%; margin: 0 auto;">
            <p style="font-size: 16px; color: #ff0;">รายชื่อทั้งหมด: {', '.join(all_names)}</p>
            <p><strong>ผู้เล่น (ปล่อยว่างเพื่อเล่นทุกคน):</strong> <br><input type="text" id="p_names" value="" placeholder="เช่น Captun, Dream" style="width: 100%; padding: 8px; margin-top: 5px; border-radius: 5px; border: none;"></p>
            <p><strong>ระดับความยาก:</strong> <br>
                <select id="p_diff" style="width: 100%; padding: 8px; margin-top: 5px; border-radius: 5px; border: none; font-size: 16px;">
                    <option value="1">ตุ่ยดุ้ย (ง่าย - 80%)</option>
                    <option value="2" selected>ซีเล็ง (ปานกลาง - 90%)</option>
                    <option value="3">สลิ้งแตก (ยาก - 95%)</option>
                </select>
            </p>
            <p><strong>เวลารวม (วินาที):</strong> <br><input type="number" id="p_time" value="60" style="width: 100%; padding: 8px; margin-top: 5px; border-radius: 5px; border: none;"></p>
            <div style="text-align: center; margin-top: 20px;">
                <button onclick="startGame()" style="padding:15px 30px; background:linear-gradient(90deg, #f0f, #0ff); color:white; font-weight:bold; border:none; cursor:pointer; font-size:22px; border-radius: 30px; box-shadow: 0 5px 15px rgba(255,0,255,0.5); text-shadow: 1px 1px 2px black;">🕹️ START GAME</button>
            </div>
        </div>
        <div id="game-view" style="display:none; position:relative; margin-top: 10px; border-radius: 10px; overflow: hidden; border: 4px solid #fff; box-shadow: 0 0 20px rgba(0,255,255,0.6);">
            <video id="video" width="640" height="480" style="display:none;" autoplay playsinline></video>
            <canvas id="canvas" width="640" height="480" style="display:block;"></canvas>
            <div id="flash" style="display:none; position:absolute; top:0; left:0; width:640px; height:480px; background:rgba(0,255,0,0.4); pointer-events: none;"></div>
        </div>
    </div>
    <script>
    async function startGame() {{
        document.getElementById('setup-menu').style.display = 'none';
        document.getElementById('game-view').style.display = 'block';
        let bgm = document.getElementById('bgm');
        if(bgm.src.length > 50) bgm.play();
        let time = document.getElementById('p_time').value;
        let diff = document.getElementById('p_diff').value;
        let names = document.getElementById('p_names').value;
        await google.colab.kernel.invokeFunction('start_game_py', [time, diff, names], {{}});
        const stream = await navigator.mediaDevices.getUserMedia({{video: true}});
        const video = document.getElementById('video');
        video.srcObject = stream;
        const canvas = document.getElementById('canvas');
        const ctx = canvas.getContext('2d');
        const captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640; captureCanvas.height = 480;
        const captureCtx = captureCanvas.getContext('2d');
        let isProcessing = false;
        setInterval(async () => {{
            if(isProcessing) return;
            isProcessing = true;
            captureCtx.drawImage(video, 0, 0, 640, 480);
            let b64 = captureCanvas.toDataURL('image/jpeg', 0.8);
            let response = await google.colab.kernel.invokeFunction('process_frame_game', [b64], {{}});
            let data = JSON.parse(response.data['text/plain'].slice(1, -1));
            ctx.save(); ctx.scale(-1, 1); ctx.drawImage(video, -640, 0, 640, 480); ctx.restore();
            if(data.game_over) {{
                ctx.fillStyle = "rgba(0,0,0,0.85)"; ctx.fillRect(0,0,640,480);
                ctx.fillStyle = "#ff0055"; ctx.font = "bold 70px Impact, sans-serif"; ctx.textAlign = "center";
                ctx.fillText("GAME OVER", 320, 220);
                ctx.fillStyle = "#0ff"; ctx.font = "bold 45px Impact, sans-serif";
                ctx.fillText("Final Score: " + data.score, 320, 300);
                bgm.pause(); return;
            }}
            if(data.passed) {{
                let flash = document.getElementById('flash');
                flash.style.display = 'block'; setTimeout(() => {{ flash.style.display = 'none'; }}, 150);
            }}
            if(data.faces) {{
                data.faces.forEach(f => {{
                    let box = f.box; let isTarget = (f.name == data.target_name);
                    ctx.strokeStyle = isTarget ? "#0f0" : "#fff"; ctx.lineWidth = 4;
                    ctx.strokeRect(box[0], box[1], box[2]-box[0], box[3]-box[1]);
                    ctx.fillStyle = isTarget ? "#0f0" : "#fff"; ctx.font = "bold 22px Arial";
                    ctx.fillText(f.name + " " + Math.round(f.prob*100) + "%", box[0], box[1]-10);
                }});
            }}
            ctx.fillStyle = "rgba(0,0,0,0.6)"; ctx.fillRect(0,0,640, 70);
            ctx.fillStyle = "rgba(0,255,255,0.3)"; ctx.fillRect(0,70,640, 5);
            ctx.fillStyle = "#0f0"; ctx.font = "bold 32px Arial"; ctx.textAlign = "left";
            ctx.fillText("TARGET: " + data.target_name, 20, 45);
            ctx.fillStyle = data.time_left < 10 ? "#ff0055" : "#fff"; ctx.font = "bold 24px Arial"; ctx.textAlign = "right";
            ctx.fillText("Time: " + data.time_left.toFixed(1) + "s", 620, 35);
            ctx.fillStyle = "#ff0"; ctx.fillText("Score: " + data.score, 620, 65);
            ctx.fillStyle = "rgba(0,0,0,0.7)"; ctx.fillRect(20, 420, 600, 35);
            let pColor = data.highest_conf >= data.confidence_needed ? "#0f0" : "#fa0";
            ctx.fillStyle = pColor; ctx.fillRect(20, 420, 600 * data.highest_conf, 35);
            ctx.strokeStyle = "#fff"; ctx.lineWidth = 2; ctx.strokeRect(20, 420, 600, 35);
            let goalX = 20 + 600 * data.confidence_needed;
            ctx.strokeStyle = "#ff0055"; ctx.lineWidth = 4; ctx.beginPath(); ctx.moveTo(goalX, 410); ctx.lineTo(goalX, 465); ctx.stroke();
            ctx.fillStyle = "#ff0055"; ctx.font = "bold 16px Arial"; ctx.textAlign = "center";
            ctx.fillText("Goal " + Math.round(data.confidence_needed*100) + "%", goalX, 405);
            ctx.textAlign = "left"; isProcessing = false;
        }}, 80);
    }}
    </script>
    '''
    display(HTML(html_code))

start_colab_game(all_names, mtcnn, resnet, clf, le, device)

